In [1]:
# LightGBM
sc.install_pypi_package("lightgbm")
sc.install_pypi_package("scikit-learn")
sc.install_pypi_package("pandas")
sc.install_pypi_package("numpy")

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
10,application_1781028837408_0011,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…



  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1781028837408_0011/container_1781028837408_0011_01_000001/tmp/spark-891c7238-24df-48d6-9d99-8760934767b8
    Can't uninstall 'python-dateutil'. No files were found to uninstall.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.3.3 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.

In [2]:

df_saved = spark.read.parquet(
    "s3://csc555-data-dd9d74e8/processed/final_features/"
)

print("Rows:", df_saved.count())
print("Columns:", len(df_saved.columns))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Rows: 992931
Columns: 24

In [3]:
feature_cols = [
    "city",
    "bd",
    "registered_via",
    "transaction_count",
    "avg_payment_plan_days",
    "avg_plan_list_price",
    "avg_actual_amount_paid",
    "auto_renew_count",
    "cancel_count",
    "log_days",
    "avg_num_25",
    "avg_num_50",
    "avg_num_75",
    "avg_num_985",
    "avg_num_100",
    "avg_num_unq",
    "avg_total_secs",
    "sum_total_secs"
]

pdf = df_saved.select(
    feature_cols + ["is_churn"]
).toPandas()

print(pdf.shape)
pdf.head()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

(992931, 19)
   city  bd  registered_via  ...  avg_total_secs  sum_total_secs  is_churn
0    14  31               7  ...     2340.949887    1.032359e+06         0
1     1   0               7  ...     9824.775080    2.957257e+06         0
2     1   0               7  ...     1961.379000    1.961379e+03         0
3    22  32               9  ...     4104.195538    7.633804e+05         0
4    13  29               9  ...    39060.542332    3.070159e+07         0

[5 rows x 19 columns]

In [4]:
from sklearn.model_selection import train_test_split

X = pdf[feature_cols]
y = pdf["is_churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

X_train: (794344, 18)
X_test: (198587, 18)

In [5]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_model.fit(X_train, y_train)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=5,
               n_estimators=300, random_state=42, subsample=0.8, verbose=-1)

In [6]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

y_pred = lgb_model.predict(X_test)
y_prob = lgb_model.predict_proba(X_test)[:, 1]

print("LightGBM Results")
print("AUC =", roc_auc_score(y_test, y_prob))
print("Accuracy =", accuracy_score(y_test, y_pred))
print("Precision =", precision_score(y_test, y_pred))
print("Recall =", recall_score(y_test, y_pred))
print("F1 =", f1_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LightGBM Results
AUC = 0.9559269322494003
Accuracy = 0.9513966170998102
Precision = 0.7373595505617978
Recall = 0.37222309752639043
F1 = 0.49471259553973407
Confusion Matrix:
[[184210   1683]
 [  7969   4725]]

In [7]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_prob = lgb_model.predict_proba(X_test)[:,1]

thresholds = [0.2,0.25,0.3,0.35,0.4,0.5]

for t in thresholds:

    y_pred_t = (y_prob >= t).astype(int)

    print("Threshold:", t)
    print("Accuracy:", accuracy_score(y_test, y_pred_t))
    print("Precision:", precision_score(y_test, y_pred_t))
    print("Recall:", recall_score(y_test, y_pred_t))
    print("F1:", f1_score(y_test, y_pred_t))
    print("--------------------")

    # LightGBM achieved the best overall performance with an AUC of 0.9559. 
    # After threshold tuning, the best threshold was 0.30, resulting in an accuracy of 0.9435, 
    # precision of 0.5531, recall of 0.6042, and F1-score of 0.5775.

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Threshold: 0.2
Accuracy: 0.9195516322820728
Precision: 0.4301166851205178
Recall: 0.7956514888923901
F1: 0.5583812472357363
--------------------
Threshold: 0.25
Accuracy: 0.9346885747808266
Precision: 0.4923128342245989
Recall: 0.6962344414684103
F1: 0.5767800039156823
--------------------
Threshold: 0.3
Accuracy: 0.9434957978115385
Precision: 0.5531117040455759
Recall: 0.6042224673073893
F1: 0.5775384962915553
--------------------
Threshold: 0.35
Accuracy: 0.9484659116659198
Precision: 0.6131346578366446
Recall: 0.5251299826689775
F1: 0.565730289399983
--------------------
Threshold: 0.4
Accuracy: 0.9505405691208387
Precision: 0.6593430980914337
Recall: 0.46809516306916654
F1: 0.5474983875426149
--------------------
Threshold: 0.5
Accuracy: 0.9513966170998102
Precision: 0.7373595505617978
Recall: 0.37222309752639043
F1: 0.49471259553973407
--------------------

In [8]:
import joblib

# save model locally first
joblib.dump(lgb_model, "/tmp/lightgbm_churn_model.pkl")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['/tmp/lightgbm_churn_model.pkl']

In [9]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "/tmp/lightgbm_churn_model.pkl",
    "csc555-data-dd9d74e8",
    "models/lightgbm_churn_model.pkl"
)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.05, max_depth=5,
               n_estimators=300, random_state=42, subsample=0.8, verbose=-1)